# Synthetic Support Ticket Generator + Gradio UI

Generate a **labeled CSV** of synthetic support tickets. Control **volume** (number of tickets), **tone** (e.g. polite, frustrated, urgent), and **categories** (e.g. billing, technical, account) from the Gradio UI. Output columns: `id`, `category`, `tone`, `subject`, `description`, `priority`.

In [1]:
import os
import json
import tempfile
from openai import OpenAI
import gradio as gr
import pandas as pd
from dotenv import load_dotenv

In [2]:
load_dotenv(override=True)

if os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY is set.")
else:
    print("Warning: OPENAI_API_KEY not set. Set it in .env or environment to use the generator.")

OPENAI_API_KEY is set.


In [3]:
TONES = ["polite", "frustrated", "neutral", "urgent"]
CATEGORIES = ["billing", "technical", "account", "refund", "product"]
PRIORITIES = ["low", "medium", "high"]
MODEL = "gpt-4o-mini"
BATCH_SIZE = 10  # tickets per API call to stay within token limits

In [4]:
def generate_support_tickets(n: int, categories: list, tone: str) -> pd.DataFrame:
    """Generate n synthetic support tickets with given categories and tone via OpenAI."""
    if not categories:
        categories = CATEGORIES.copy()
    client = OpenAI()
    all_rows = []
    ticket_id = 1
    remaining = n
    while remaining > 0:
        batch_n = min(remaining, BATCH_SIZE)
        categories_str = ", ".join(categories)
        system_prompt = (
            "You generate synthetic customer support tickets. "
            "Reply with a single JSON array only, no other text. "
            "Each object must have: id (integer, 1-based in this batch), category (one of: " + categories_str + "), "
            "tone (customer tone: " + tone + "), subject (short title), description (1-3 sentences), "
            "priority (one of: low, medium, high)."
        )
        user_prompt = f"Generate exactly {batch_n} support ticket(s). Use categories from: {categories_str}. Customer tone: {tone}."
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                max_tokens=2048,
                temperature=0.7,
            )
            content = response.choices[0].message.content.strip()
            if content.startswith("```"):
                content = content.split("```")[1]
                if content.startswith("json"):
                    content = content[4:]
            data = json.loads(content)
            if not isinstance(data, list):
                data = [data]
            for t in data:
                all_rows.append({
                    "id": ticket_id,
                    "category": t.get("category", ""),
                    "tone": t.get("tone", tone),
                    "subject": t.get("subject", ""),
                    "description": t.get("description", ""),
                    "priority": t.get("priority", "medium"),
                })
                ticket_id += 1
        except (json.JSONDecodeError, Exception) as e:
            all_rows.append({
                "id": ticket_id, "category": "error", "tone": tone,
                "subject": "Parse error", "description": str(e), "priority": "high",
            })
            ticket_id += 1
        remaining -= batch_n
    return pd.DataFrame(all_rows)

In [5]:
def run_generator(volume, categories: list, tone: str):
    """Generate tickets and return (DataFrame for preview, CSV file path for download)."""
    n = int(volume) if volume is not None else 0
    if n < 1:
        return pd.DataFrame(), None
    df = generate_support_tickets(n, categories or CATEGORIES, tone or "neutral")
    fd, path = tempfile.mkstemp(suffix=".csv")
    os.close(fd)
    try:
        df.to_csv(path, index=False)
        return df, path
    except Exception:
        os.remove(path)
        raise


with gr.Blocks(title="Synthetic Support Ticket Generator") as demo:
    gr.Markdown("## Generate labeled support tickets")
    with gr.Row():
        volume = gr.Slider(1, 50, value=10, step=1, label="Number of tickets")
        tone = gr.Dropdown(choices=TONES, value="neutral", label="Customer tone")
    categories = gr.CheckboxGroup(choices=CATEGORIES, value=CATEGORIES, label="Categories")
    generate_btn = gr.Button("Generate", variant="primary")
    gr.Markdown("### Preview")
    preview_df = gr.Dataframe(label="Tickets")
    download_file = gr.File(label="Download CSV", interactive=False)
    generate_btn.click(
        fn=run_generator,
        inputs=[volume, categories, tone],
        outputs=[preview_df, download_file],
    )

Matplotlib is building the font cache; this may take a moment.


In [6]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
